In [ ]:
!pip install feedparser

"""
구글 뉴스 RSS 수집기 — "LLM의 마법 주문서: Automate Anything" 뉴스 분석 시연용

사용법:
    pip install feedparser
    python google_news_rss.py                      # 기본: "인공지능"
    python google_news_rss.py 생성형AI 반도체        # 키워드 여러 개
    python google_news_rss.py 인공지능 --days 1     # 최근 1일만

결과:
    news_YYYYMMDD_HHMM.csv  (엑셀에서 바로 열림)
    news_YYYYMMDD_HHMM.md   (AI에 그대로 붙여 넣는 용도)
"""
import argparse
import csv
import sys
from datetime import datetime, timedelta, timezone
from urllib.parse import quote

import feedparser

KST = timezone(timedelta(hours=9))


def fetch(keyword: str, days: int):
    url = (
        "https://news.google.com/rss/search?"
        f"q={quote(keyword)}+when:{days}d&hl=ko&gl=KR&ceid=KR:ko"
    )
    feed = feedparser.parse(url)
    rows = []
    for e in feed.entries:
        # 제목 끝의 " - 매체명"을 분리
        title = e.get("title", "")
        source = e.get("source", {}).get("title", "")
        if source and title.endswith(f" - {source}"):
            title = title[: -len(f" - {source}")]
        published = ""
        if e.get("published_parsed"):
            published = (
                datetime(*e.published_parsed[:6], tzinfo=timezone.utc)
                .astimezone(KST)
                .strftime("%Y-%m-%d %H:%M")
            )
        rows.append(
            {
                "keyword": keyword,
                "published_kst": published,
                "source": source,
                "title": title,
                "link": e.get("link", ""),
            }
        )
    return rows


def main():
    p = argparse.ArgumentParser()
    p.add_argument("keywords", nargs="*", default=["인공지능"])
    p.add_argument("--days", type=int, default=1)
    args = p.parse_args()

    all_rows, seen = [], set()
    for kw in args.keywords:
        for r in fetch(kw, args.days):
            if r["title"] in seen:  # 같은 기사 중복 제거
                continue
            seen.add(r["title"])
            all_rows.append(r)

    if not all_rows:
        print("받아온 기사가 없습니다. 네트워크나 키워드를 확인하세요.")
        sys.exit(1)

    all_rows.sort(key=lambda r: r["published_kst"], reverse=True)
    stamp = datetime.now(KST).strftime("%Y%m%d_%H%M")

    csv_path = f"news_{stamp}.csv"
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
        w.writeheader()
        w.writerows(all_rows)

    md_path = f"news_{stamp}.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(f"# 구글 뉴스 RSS 수집 ({stamp} KST)\n")
        f.write(f"키워드: {', '.join(args.keywords)} / 최근 {args.days}일 / {len(all_rows)}건\n\n")
        for i, r in enumerate(all_rows, 1):
            f.write(f"{i}. [{r['published_kst']}] {r['title']} — {r['source']}\n   {r['link']}\n")

    print(f"{len(all_rows)}건 저장: {csv_path}, {md_path}")


if __name__ == "__main__":
    # 노트북(Colab) 환경에서 인수 없이 실행되도록 임시 방편 처리 (sys.argv 초기화)
    sys.argv = [sys.argv[0]]
    main()

In [ ]:
import glob
from google.colab import files

# 가장 최근에 생성된 .md 파일과 .csv 파일 찾기
md_files = sorted(glob.glob('news_*.md'))
csv_files = sorted(glob.glob('news_*.csv'))

if md_files:
    latest_md = md_files[-1]
    print(f'다운로드할 파일: {latest_md}')
    files.download(latest_md)
else:
    print('다운로드할 .md 파일이 존재하지 않습니다. 먼저 위의 뉴스 수집기 셀을 실행해 주세요.')

# CSV 파일도 다운로드하고 싶으시다면 아래 주석을 해제하고 실행하세요.
# if csv_files:
#     files.download(csv_files[-1])